**Лабораторный практикум по курсу «Распознавание диктора», Университет ИТМО, 2021**		

**Лабораторная работа №3. Построение дикторских моделей и их сравнение**

**Цель работы:** изучение процедуры построения дикторских моделей с использованием глубоких нейросетевых архитектур.

**Краткое описание:** в рамках настоящей лабораторной работы предлагается изучить и реализовать схему построения дикторских моделей с использованием глубокой нейросетевой архитектуры, построенной на основе ResNet-блоков. Процедуры обучения и тестирования предлагается рассмотреть по отношению к задаче идентификации на закрытом множестве, то есть для ситуации, когда дикторские классы являются строго заданными. Тестирование полученной системы предполагает использование доли правильных ответов (accuracy) в качестве целевой метрики оценки качества.

**Данные:** в качестве данных для выполнения лабораторной работы предлагается использовать базу [VoxCeleb1](http://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html).

**Содержание лабораторной работы**

1. Подготовка данных для обучения и тестирования блока построения дикторских моделей.							

2. Обучение параметров блока построения дикторских моделей без учёта процедуры аугментации данных.

3. Обучение параметров блока построения дикторских моделей с учётом процедуры аугментации данных.

4. Тестированное блока построения дикторских моделей.

In [1]:
# IPython extension to reload modules before executing user code
%load_ext autoreload
%autoreload 2

# Import of modules
import os
import sys

sys.path.append(os.path.realpath('..'))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from common import download_dataset, concatenate, extract_dataset, part_extract, download_protocol, split_musan
from ResNetBlocks import BasicBlock
from LossFunction import AAMSoftmaxLoss
from Optimizer import SGDOptimizer
from Scheduler import OneCycleLRScheduler
from load_save_pth import saveParameters, loadParameters

/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaconda/lib/python3.10/site-packages/scipy/__init__.py:132: UserWarning: A NumPy version >=1.21.6 and <1.28.0 is required for this version of SciPy (detected version 2.2.6)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaconda/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaco

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

**1. Подготовка данных для обучения и тестирования детектора речевой активности**

В ходе выполнения лабораторной работы необходимы данные для выполнения процедуры обучения и процедуры тестирования нейросетевого блока генерации дикторских моделей. Возьмём в качестве этих данных звукозаписи, сохраненные в формат *wav*, из корпуса [VoxCeleb1 dev set](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html). Данный корпус содержит 148,642 звукозаписи (частота дискретизации равна 16кГц) для 1,211 дикторов женского и мужского пола, разговаривающих преимущественно на английском языке.

В рамках настоящего пункта требуется выполнить загрузку и распаковку звуковых wav-файлов из корпуса VoxCeleb1 dev set.

![Рисунок 1](https://analyticsindiamag.com/wp-content/uploads/2020/12/image.png "VoxCeleb. Крупномасштабная аудиовизуальная база данных человеческой речи.")

In [7]:
# # Download VoxCeleb1 (test set)
# with open('../data/lists/datasets.txt', 'r') as f:
#     lines = f.readlines()

# download_dataset(lines, user='voxceleb1902', password='nx0bl2v2', save_path='../data')

In [8]:
# # Concatenate archives for VoxCeleb1 dev set
# with open('../data/lists/concat_arch.txt', 'r') as f:
#     lines = f.readlines()
    
# concatenate(lines, save_path='../data')

In [9]:
# # Extract VoxCeleb1 dev set
# extract_dataset(save_path='../data/voxceleb1_dev', fname='../data/vox1_dev_wav.zip')

In [10]:
# # Download VoxCeleb1 identification protocol
# with open('../data/lists/protocols.txt', 'r') as f:
#     lines = f.readlines()
    
# download_protocol(lines, save_path='../data/voxceleb1_test')

**2. Обучение параметров блока построения дикторских моделей без учёта процедуры аугментации данных**

Построение современных дикторских моделей, как правило, выполняется с использованием нейросетевых архитектур, многие из которых позаимствованы из области обработки цифровых изображений. Одними из наиболее распространенных нейросетевых архитектур, используемыми для построения дикторских моделей, являются [ResNet-подобные архитектуры](https://arxiv.org/pdf/1512.03385.pdf). В рамках настоящего пункта предлагается выполнить адаптацию нейросетевой архитектуры ResNet34 для решения задачи генерации дикторских моделей (дикторских эмбеддингов). *Дикторский эмбеддинг* – это высокоуровневый вектор-признаков, состоящий, например, из 128, 256 и т.п. значений, содержащий особенности голоса конкретного человека. При решении задачи распознавания диктора можно выделить эталонные и тестовые дикторские эмбеддинги. *Эталонные эмбеддинги* формируются на этапе регистрации дикторской модели определённого человека и находятся в некотором хранилище данных. *Тестовые эмбеддинги* формируются на этапе непосредственного использования системы голосовой биометрии на практике, когда некоторый пользователь пытается получить доступ к соответствующим ресурсам. Система голосовой биометрии сравнивает по определённой метрике эталонные и тестовые эмбеддинги, формируя оценку сравнения, которая, после её обработки блоком принятия решения, позволяет сделать вывод о том, эмбеддинги одинаковых или разных дикторов сравниваются между собой.

Адаптация различных нейросетевых архитектур из обработки изображений к решению задачи построения дикторских моделей является непростой задачей. Возьмём за основу готовое решение, предложенной в рамках [следующей публикации](https://arxiv.org/pdf/2002.06033.pdf) и адаптируем его применительно к выполнению настоящей лабораторной работы.

Необходимо отметить, что построение дикторских моделей, как правило, требует наличия *акустических признаков*, вычисленных для звукозаписей тренировочной, валидационной и тестовой баз данных. В качестве примера подобных признаков в рамках настоящей лабораторной работы воспользуемся *логарифмами энергий на выходе мел-банка фильтров*. Важно отметить, что акустические признаки подвергаются некоторым процедурам предобработки перед их непосредственной передачей в блок построения дикторских моделей. В качестве этих процедур можно выделить: нормализация и масштабирование признаков, сохранение только речевых фреймов на основе разметки детектора речевой активности и т.п.

После того, как акустические признаки подготовлены, они могут быть переданы на блок построения дикторских моделей. Как правило, структура современных дикторских моделей соответствует структуре [x-векторных архитектур](https://www.danielpovey.com/files/2018_icassp_xvectors.pdf). Эти архитектуры состоят из четырёх ключевых элементов: 

1. **Фреймовый уровень.** Предназначен для формирования локальных представлений голоса конкретного человека. На этом уровне как раз и применяются нейросетевые архитектуры на базе свёрточных нейронных сетей, например, ResNet, позволяющих с использованием каскадной схемы из множества фильтров с локальной маской захватить некоторый локальный контекст шаблона голоса человека. Выходом фреймового уровня является набор высокоуровневых представлений (карт-признаков), содержащих локальные особенности голоса человека.

2. **Уровень статистического пулинга** позволяет сформировать промежуточный вектор-признаков, фиксированной длины, которая является одинаковой для звукозаписи любой длительности. В ходе работы блока статистического пулинга происходит удаление временной размерности, присутствующей в картах-признаков. Это достигается путём выполнения процедуры усреднения карт-признаков вдоль оси времени. Выходом уровня статистического пулинга являются вектор среднего и вектор среднеквадратического отклонения, вычисленные на основе карт-признаков. Эти вектора конкатенируются и передаются для дальнейшей обработки на сегментом уровне.

3. **Сегментный уровень.** Предназначен для трансформации промежуточного вектора, как правило, высокой размерности, в компактный вектор-признаков, представляющий собой дикторский эмбеддинг. Необходимо отметить, что на сегментном уровне расположены один или несколько полносвязных нейросетевых слоёв, а обработка данных выполняется по отношению ко всей звукозаписи, а не только к некоторому её локальному контексту, как на фреймовом уровне.

4. **Уровень выходного слоя.** Представляет полносвязный слой с softmax-функциями активации. Количество активаций равно числу дикторов в тренирочной выборке. На вход выходноя слоя подаётся дикторский эмбеддинг, а на выходе – формируется набор апостериорных вероятностей, определяющих принадлежность эмбеддинга к одному из дикторских классов в тренировочной выборке. Необходимо отметить, что, как правило, в современных нейросетевых системах построения дикторских моделей выходной используется только на этапе обучения параметров и на этапе тестирования не используется (на этапе тестирования используются только три первых уровня архитектуры).

Обучение модели генерации дикторских эмбеддингов выполняется путём решения задачи *классификации* или, выражаясь терминами из области биометрии, *идентификации на закрытом множестве* (количество дикторских меток является строго фиксированным). В качестве используемой стоимостной функции выступает *категориальная кросс-энтропия*. Обучение выполняется с помощью мини-батчей, содержащих короткие фрагменты карт акустических признаков (длительностью несколько секунд) различных дикторов из тренировочной базы данных. Обучение на коротких фрагментов позволяет избежать сильного переобучения нейросетевой модели. При выполнении процедуры обучения требуется подобрать набор гиперпараметров, выбрать обучения и метод численной оптимизации.

Для успешного выполнения настоящего пункта необходимо сделать следующее:

1. Сгенерировать списки тренировочных, валидационных и тестовых данных на основе идентификационного протокола базы VoxCeleb1, содержащегося в файле **../data/voxceleb1_test/iden_split.txt**. При генерации списков требуется исключить из них звукозаписи дикторов, которые входят в базу [VoxCeleb1 test set](https://thor.robots.ox.ac.uk/~vgg/data/voxceleb/vox1a/vox1_test_wav.zip). Это позволит выполнить тестирования обученных блоков генерации дикторских моделей на протоколе [VoxCeleb1-O cleaned](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt), который составлен по отношению к данным из VoxCeleb1 test set, в лабораторной работе №4.

2. Инициализировать обучаемую дикторскую модель, выбрав любой возможный вариант её архитектуры, предлагаемый в рамках лабораторной работы. При реализации блока статистического пулинга предлагается выбрать либо его классический вариант, предложенный в [следующей работе](https://www.danielpovey.com/files/2018_icassp_xvectors.pdf), либо его более продвинутую версию основанную на использовании [механизмов внимания](https://arxiv.org/pdf/1803.10963.pdf). Использование последней версии статистического пулинга позволяет реализовать детектор речевой активности прямо внутри блока построения дикторских моделей.

3. Инициализировать загрузчики тренировочной и валидационной выборки.

4. Инициализировать оптимизатор и планировщик для выполнения процедуры обучения.

5. Описать процедуру валидации/тестирования блока построения дикторских моделей.

6. Описать процедуру обучения и запустить её, контролируя значения стоимостной функции и доли правильных ответов на тренировочном множестве, а также долю правильных ответов на валидационном множестве.

In [2]:
# Select hyperparameters

# Acoustic features
n_mels            = 40                                   # number of mel filters in bank filters
log_input         = True                                 # logarithm of features by level

# Neural network archtecture
layers            = [3, 4, 6, 3]                         # number of ResNet blocks in different level of frame level
activation        = nn.ReLU                              # activation function used in ResNet blocks
num_filters       = [32, 64, 128, 256]                   # number of filters of ResNet blocks in different level of frame level
encoder_type      = 'SP'                                 # type of statistic pooling layer ('SP'  – classical statistic pooling 
                                                         # layer and 'ASP' – attentive statistic pooling)
nOut              = 512                                  # embedding size

# Loss function for angular losses
margin            = 0.35                                 # margin parameter
scale             = 32.0                                 # scale parameter

# Train dataloader
max_frames_train  = 200                                  # number of frame to train
train_path        = '../data/voxceleb1_dev/wav'          # path to train wav files
batch_size_train  = 128                                  # batch size to train
pin_memory        = False                                # pin memory
num_workers_train = 5                                    # number of workers to train
shuffle           = True                                 # shuffling of training examples

# Validation dataloader
max_frames_val    = 1000                                 # number of frame to validate
val_path          = '../data/voxceleb1_dev/wav'          # path to val wav files
batch_size_val    = 128                                  # batch size to validate
num_workers_val   = 5                                    # number of workers to validate

# Test dataloader
max_frames_test   = 1000                                 # number of frame to test
test_path         = '../data/voxceleb1_dev/wav'          # path to val wav files
batch_size_test   = 128                                  # batch size to test
num_workers_test  = 5                                    # number of workers to test

# Optimizer
lr                = 2.5                                  # learning rate value
weight_decay      = 0                                    # weight decay value

# Scheduler
val_interval      = 5                                    # frequency of validation step
max_epoch         = 40                                   # number of epoches

# Augmentation
musan_path        = '../data/musan_split'                # path to splitted SLR17 dataset
rir_path          = '../data/RIRS_NOISES/simulated_rirs' # path to SLR28 dataset

In [3]:
# Generate data lists
train_list = []
val_list   = []
test_list  = []

with open('../data/voxceleb1_test/iden_split.txt', 'r') as f:
    lines = f.readlines()
    
black_list = os.listdir('../data/voxceleb1_test/wav')   # exclude speaker IDs from VoxCeleb1 test set
num_train_spk = []                                      # number of train speakers

for line in lines:
    line   = line.strip().split(' ')
    spk_id = line[1].split('/')[0]
    
    if not (spk_id in black_list):
        num_train_spk.append(spk_id)
        
    else:
        continue
    
    # Train list
    if (line[0] == '1'):
        train_list.append(' '.join([spk_id, line[1]]))
    
    # Validation list
    elif (line[0] == '2'):
        val_list.append(' '.join([spk_id, line[1]]))
    
    # Test list
    elif (line[0] == '3'):
        test_list.append(' '.join([spk_id, line[1]]))
        
num_train_spk = len(set(num_train_spk))

In [17]:
import numpy

[autoreload of numpy.lib.format failed: Traceback (most recent call last):
  File "/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaconda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaconda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
  File "/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaconda/lib/python3.10/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 619, in _exec
  File "<frozen importlib._bootstrap_external>", line 883, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaconda/lib/python3.10/site-packages/numpy/lib/format.py", line 167, in <module>
    from numpy.lib.utils import safe_eval

In [12]:
numpy.__version__

'2.2.6'

In [15]:
pip uninstall numpy -y

Found existing installation: numpy 1.21.1
Uninstalling numpy-1.21.1:
  Successfully uninstalled numpy-1.21.1
You can safely remove it manually.
Note: you may need to restart the kernel to use updated packages.


In [16]:
pip install numpy==1.21.1

Looking in indexes: https://nid-artifactory.ad.speechpro.com/artifactory/api/pypi/pypi/simple
  Using cached numpy-1.21.1-cp310-cp310-linux_x86_64.whl
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tts 0.22.0 requires numpy==1.22.0; python_version <= "3.10", but you have numpy 1.21.1 which is incompatible.
chromadb 0.5.23 requires numpy>=1.22.5, but you have numpy 1.21.1 which is incompatible.
chromadb 0.5.23 requires tokenizers<=0.20.3,>=0.13.2, but you have tokenizers 0.22.1 which is incompatible.
faiss-cpu 1.9.0.post1 requires numpy<3.0,>=1.25.0, but you have numpy 1.21.1 which is incompatible.
graph-retriever 0.6.1 requires numpy>=1.26.4, but you have numpy 1.21.1 which is incompatible.
jax 0.6.2 requires numpy>=1.26, but you have numpy 1.21.1 which is incompatible.
jax 0.6.2 requires scipy>=1.12, but you have scipy 1.11.4 which is incompatible.
jaxlib 0.6

In [11]:
from exercises_blank import ResNet, MainModel


A module that was compiled using NumPy 1.x cannot be run in
NumPy 1.21.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaconda/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaconda/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/mnt/asr_hot/dutov/cryfish/audiollm2interspeech/anaconda/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/mnt/asr_hot/dutov/cryfish/audiollm2in

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [5]:
# Initialize model
model      = ResNet(BasicBlock, layers=layers, activation=activation, num_filters=num_filters, nOut=nOut, encoder_type=encoder_type, n_mels=n_mels, log_input=log_input)
trainfunc  = AAMSoftmaxLoss(nOut=nOut, nClasses=num_train_spk, margin=margin, scale=scale)
main_model = MainModel(model, trainfunc).cuda()

Embedding size is 512, encoder SP.
Initialised AAM softmax margin 0.350 scale 32.000.


In [6]:
from exercises_blank import train_dataset_loader, test_dataset_loader

In [7]:
# Initialize train dataloader (without augmentation)
train_dataset = train_dataset_loader(train_list=train_list, max_frames=max_frames_train, train_path=train_path)
train_loader  = DataLoader(train_dataset, batch_size=batch_size_train, pin_memory=pin_memory, num_workers=num_workers_train, shuffle=shuffle)

# Initialize validation dataloader
val_dataset = test_dataset_loader(test_list=val_list, max_frames=max_frames_val, test_path=val_path)
val_loader  = DataLoader(val_dataset, batch_size=batch_size_val, num_workers=num_workers_val)

In [8]:
# Initialize optimizer and scheduler
optimizer = SGDOptimizer(main_model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = OneCycleLRScheduler(optimizer, 
                                pct_start=0.30, 
                                cycle_momentum=False, 
                                max_lr=lr, 
                                div_factor=20, 
                                final_div_factor=10000, 
                                total_steps=max_epoch*len(train_loader))

Initialised SGD optimizer.
Initialised OneCycle LR scheduler.


In [35]:
start_epoch = 0
checkpoint_flag = False

if checkpoint_flag:
    start_epoch = loadParameters(main_model, optimizer, scheduler, path='../data/lab3_models/lab3_model_0004.pth')
    start_epoch = start_epoch + 1

# Train model
for num_epoch in range(start_epoch, max_epoch):
    train_loss, train_top1 = train_network(train_loader, main_model, optimizer, scheduler, num_epoch, verbose=True)
    
    print("Epoch {:1.0f}, Loss (train set) {:f}, Accuracy (train set) {:2.3f}%".format(num_epoch, train_loss, train_top1))

    if (num_epoch + 1)%val_interval == 0:
        _, val_top1 = test_network(val_loader, main_model)
        
        print("Epoch {:1.0f}, Accuracy (validation set) {:2.3f}%".format(num_epoch, val_top1))
        
        saveParameters(main_model, optimizer, scheduler, num_epoch, path='../data/lab3_models')

/mnt/asr_hot/dutov/study/dictor_ver/dictor_verification_lab/lab3/exercises_blank.py:209: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


Epoch 0, Batch 1, LR 0.125000 Loss 19.177011, Accuracy 0.000%
Epoch 0, Batch 2, LR 0.125000 Loss 19.024094, Accuracy 0.000%
Epoch 0, Batch 3, LR 0.125000 Loss 19.014029, Accuracy 0.260%
Epoch 0, Batch 4, LR 0.125000 Loss 18.995182, Accuracy 0.195%
Epoch 0, Batch 5, LR 0.125001 Loss 19.058317, Accuracy 0.156%
Epoch 0, Batch 6, LR 0.125001 Loss 19.166916, Accuracy 0.130%
Epoch 0, Batch 7, LR 0.125001 Loss 19.193203, Accuracy 0.112%
Epoch 0, Batch 8, LR 0.125002 Loss 19.262180, Accuracy 0.098%
Epoch 0, Batch 9, LR 0.125002 Loss 19.310120, Accuracy 0.174%
Epoch 0, Batch 10, LR 0.125003 Loss 19.323282, Accuracy 0.156%
Epoch 0, Batch 11, LR 0.125004 Loss 19.380438, Accuracy 0.142%
Epoch 0, Batch 12, LR 0.125004 Loss 19.429214, Accuracy 0.130%
Epoch 0, Batch 13, LR 0.125005 Loss 19.468081, Accuracy 0.120%
Epoch 0, Batch 14, LR 0.125006 Loss 19.550341, Accuracy 0.112%
Epoch 0, Batch 15, LR 0.125007 Loss 19.601093, Accuracy 0.104%
Epoch 0, Batch 16, LR 0.125008 Loss 19.665651, Accuracy 0.098%
E

**3. Обучение параметров блока построения дикторских моделей с учётом процедуры аугментации данных**

Известно, что рроцедуры формирования и передачи речевого сигнала могут сопровождаться воздействием шумов и помех, приводящих к искажению сигнала. В качестве примеров искажающих факторов, влияющих на ухудшение качестве речевого сигнала можно привести: импульсный отклик помещения (реверберация), фоновый шум голосов группы нецелевых дикторов, звук телевизора или радиоприёмника и т.п. Разработка конвейера системы голосовой биометрии требует учёта воздействия искажающих факторов на качество её работы. Поскольку процедура построения современных дикторских моделей основана на обучении глубоких нейронных сетей, требующих большие объёмы данных для обучения их параметров, возможным вариантом увеличения тренировочной выборки может являться использование методов аугментации статистических данных. *Аугментация* – методика создания дополнительных обучающих примеров из имеющихся данных путём внесения в них искажений, которые могут потенциально возникнуть на этапе итогового тестирования системы.

Как правило, при решении задачи аугментации данных в речевой обработке используются дополнительные базы шумов и помех. В качестве примеров можно привести базы [SLR17](https://openslr.org/17/) (корпус музыкальных, речевых и шумовых звукозаписей) и [SLR28](https://openslr.org/28/) (база данных реальных и симулированных импульсных откликов комнат, а также изотропных и точечных шумов). Важно отметить, что перед применением с использованием методов аугментации подобных баз к имеющимся данным, требуется убедиться, что частоты дискретизации искажающих баз и оригинальных данных являются одинаковыми. Применительно к рассматриваемому лабораторному практикуму частоты дискретизации всех используемых звукозаписей должны быть равными 16кГц.

Как известно, можно выделить два режима аугментации данных: *онлайн* (применяется в ходе процедуры обучения) и *оффлайн* (применяется до процедуры обучения) аугментацию. В рамках настоящей лабораторной работы предлагается использовать онлайн аугментацию в силу не очень большого набора тренировочных данных и большей гибкости экспериментов, чем вс случае онлайн аугментации. Необходимо отметить, что применение онлайн аугментации на практике замедляет процедуру обучения, по сравнению с оффлайн аугментацией, так как наложение искажений, извлечение акустических признаков и их возможная предобработка требует определённого машинного времени.

В рамках настоящего пункта предлагается сделать следующее:

1. Загрузить и извлечь данные из базы SLR17 (MUSAN). Частота дискретизации данных в рассматриваемой базе равна 16кГц по умолчанию. Поскольку звукозаписи рассматриваемой базы являются достаточно длинными, рекомендуется предварительно разбить эту базу на более маленькие фрагменты (например, длительностью 5 секунд с шагом 3 секунды), сохранив их на диск. 

2. Загрузить и извлечь данные из базы SLR28 (MUSAN). Частота дискретизации данных в рассматриваемой базе равна 16кГц по умолчанию.

3. Модернизировать загрузчик тренировочных данных под возможность случайного наложения (искажаем исходные звукозаписи) и не наложения (не искажаем исходные звукозаписи) одного из четырёх типов искажений (реверберация, музыкальный шум, фоновый шум голосов нескольких дикторов, неструктурированный шум), описанных внутри класса **AugmentWAV** следующего программного кода: **../common/DatasetLoader.py**.

4. Используя процедуру обучения из предыдущего пункта с идентичными настройками выполнить тренировку параметров блока генерации дикторских моделей на исходных данных при наличии их аугментирвоанных копий.

In [17]:
# # Download SLR17 (MUSAN) and SLR28 (RIR noises) datasets
# with open('../data/lists/augment_datasets.txt', 'r') as f:
#     lines = f.readlines()
    
# download_dataset(lines, user=None, password=None, save_path='../data')

In [ ]:
# # Extract SLR17 (MUSAN)
# extract_dataset(save_path='../data', fname='../data/musan.tar.gz')

# # Extract SLR28 (RIR noises)
# part_extract(save_path='../data', fname='../data/rirs_noises.zip', target=['RIRS_NOISES/simulated_rirs/mediumroom', 'RIRS_NOISES/simulated_rirs/smallroom'])

Extracting of ../data/musan.tar.gz is successful.
Extracting ../data/rirs_noises.zip


In [11]:
# # Split MUSAN (SLR17) dataset for faster random access
# split_musan(save_path='../data')

0 ../data/musan/music/fma/music-fma-0001.wav
1 ../data/musan/music/fma/music-fma-0002.wav
2 ../data/musan/music/fma/music-fma-0003.wav
3 ../data/musan/music/fma/music-fma-0004.wav
4 ../data/musan/music/fma/music-fma-0005.wav
5 ../data/musan/music/fma/music-fma-0006.wav
6 ../data/musan/music/fma/music-fma-0007.wav
7 ../data/musan/music/fma/music-fma-0008.wav
8 ../data/musan/music/fma/music-fma-0009.wav
9 ../data/musan/music/fma/music-fma-0010.wav
10 ../data/musan/music/fma/music-fma-0011.wav
11 ../data/musan/music/fma/music-fma-0012.wav
12 ../data/musan/music/fma/music-fma-0120.wav
13 ../data/musan/music/fma/music-fma-0122.wav
14 ../data/musan/music/fma/music-fma-0124.wav
15 ../data/musan/music/fma/music-fma-0013.wav
16 ../data/musan/music/fma/music-fma-0014.wav
17 ../data/musan/music/fma/music-fma-0015.wav
18 ../data/musan/music/fma/music-fma-0016.wav
19 ../data/musan/music/fma/music-fma-0017.wav
20 ../data/musan/music/fma/music-fma-0018.wav
21 ../data/musan/music/fma/music-fma-0019.wa

In [12]:
# Initialize model
model      = ResNet(BasicBlock, layers=layers, activation=activation, num_filters=num_filters, nOut=nOut, encoder_type=encoder_type, n_mels=n_mels, log_input=log_input)
trainfunc  = AAMSoftmaxLoss(nOut=nOut, nClasses=num_train_spk, margin=margin, scale=scale)
main_model = MainModel(model, trainfunc).cuda()

Embedding size is 512, encoder SP.
Initialised AAM softmax margin 0.350 scale 32.000.


In [13]:
# Initialize train dataloader (without augmentation)
train_dataset = train_dataset_loader(train_list=train_list, 
                                     max_frames=max_frames_train, 
                                     train_path=train_path, 
                                     augment=True, 
                                     musan_path=musan_path, 
                                     rir_path=rir_path)

train_loader  = DataLoader(train_dataset, batch_size=batch_size_train, pin_memory=pin_memory, num_workers=num_workers_train, shuffle=shuffle)

# Initialize validation dataloader
val_dataset = test_dataset_loader(test_list=val_list, max_frames=max_frames_val, test_path=val_path)
val_loader  = DataLoader(val_dataset, batch_size=batch_size_val, num_workers=num_workers_val)

In [14]:
# Initialize optimizer and scheduler
optimizer = SGDOptimizer(main_model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = OneCycleLRScheduler(optimizer, 
                                pct_start=0.30, 
                                cycle_momentum=False, 
                                max_lr=lr, 
                                div_factor=20, 
                                final_div_factor=10000, 
                                total_steps=max_epoch*len(train_loader))

Initialised SGD optimizer.
Initialised OneCycle LR scheduler.


In [15]:
from exercises_blank import train_network, test_network

In [16]:
start_epoch = 0
checkpoint_flag = False

if checkpoint_flag:
    start_epoch = loadParameters(main_model, optimizer, scheduler, path='../data/lab3_models_aug/lab3_model_0004.pth')
    start_epoch = start_epoch + 1

# Train model
for num_epoch in range(start_epoch, max_epoch):
    train_loss, train_top1 = train_network(train_loader, main_model, optimizer, scheduler, num_epoch, verbose=True)
    
    print("Epoch {:1.0f}, Loss (train set) {:f}, Accuracy (train set) {:2.3f}%".format(num_epoch, train_loss, train_top1))

    if (num_epoch + 1)%val_interval == 0:
        _, val_top1 = test_network(val_loader, main_model)
        
        print("Epoch {:1.0f}, Accuracy (validation set) {:2.3f}%".format(num_epoch, val_top1))
        
        saveParameters(main_model, optimizer, scheduler, num_epoch, path='../data/lab3_models_aug')

/mnt/asr_hot/dutov/study/dictor_ver/dictor_verification_lab/lab3/exercises_blank.py:209: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


Epoch 0, Batch 1, LR 0.125000 Loss 19.103634, Accuracy 0.000%
Epoch 0, Batch 2, LR 0.125000 Loss 19.050449, Accuracy 0.000%
Epoch 0, Batch 3, LR 0.125000 Loss 19.005800, Accuracy 0.000%
Epoch 0, Batch 4, LR 0.125000 Loss 19.034309, Accuracy 0.000%
Epoch 0, Batch 5, LR 0.125001 Loss 19.057117, Accuracy 0.000%
Epoch 0, Batch 6, LR 0.125001 Loss 19.078058, Accuracy 0.000%
Epoch 0, Batch 7, LR 0.125001 Loss 19.133162, Accuracy 0.000%
Epoch 0, Batch 8, LR 0.125002 Loss 19.208216, Accuracy 0.000%
Epoch 0, Batch 9, LR 0.125002 Loss 19.326334, Accuracy 0.000%
Epoch 0, Batch 10, LR 0.125003 Loss 19.476908, Accuracy 0.000%
Epoch 0, Batch 11, LR 0.125004 Loss 19.601665, Accuracy 0.071%
Epoch 0, Batch 12, LR 0.125004 Loss 19.765394, Accuracy 0.065%
Epoch 0, Batch 13, LR 0.125005 Loss 19.864031, Accuracy 0.120%
Epoch 0, Batch 14, LR 0.125006 Loss 19.969868, Accuracy 0.112%
Epoch 0, Batch 15, LR 0.125007 Loss 20.142132, Accuracy 0.104%
Epoch 0, Batch 16, LR 0.125008 Loss 20.221951, Accuracy 0.098%
E

**4. Тестирование блока построения дикторских моделей**

Из литературы известно, что применение алгоритмов машинного обучения на практике требует использования трёх наборов данных: *тренировочное множество* (используется для обучения параметров модели), *валидационное множество* (используется для настройки гиперпараметров), *тестовое множество* (используется для итогового тестирования).

В рамках настоящего пункта предлагается выполнить итоговое тестирования блоков генерации дикторских моделей, обученных без аугментации и с аугментацией тренировочных данных, и сравнить полученные результаты. При проведении процедуры тестирования рекомендуется выбрать различное количество фреймов для тестовых звукозаписей, чтобы грубо понять то, как длительность фонограммы влияет на качество распознавания диктора.

В качестве целевой метрики предлагается использовать *долю правильных ответов*, то есть количество верно классифицированных объектов по отношению к общему количеству объектов тестового множества. Как и при проведении процедуры обучения и валидации, рассматриваемая процедура тестирования предполагает решение задачи идентификации диктора на закрытом множестве.

In [18]:
# Initialize test dataloader
test_dataset = test_dataset_loader(test_list=test_list, max_frames=max_frames_test, test_path=test_path)
test_loader = DataLoader(test_dataset, batch_size=batch_size_test, num_workers=num_workers_test)

In [19]:
# Load model without augmentation
num_epoch = loadParameters(main_model, optimizer, scheduler, path='../data/lab3_models/lab3_model_0039.pth')

# Test model
_, test_top1 = test_network(test_loader, main_model)

print("Epoch {:1.0f}, Accuracy (test set) {:2.3f}%".format(num_epoch, test_top1))

Epoch 39, Accuracy (test set) 96.260%


In [20]:
# Load model with augmentation
num_epoch = loadParameters(main_model, optimizer, scheduler, path='../data/lab3_models_aug/lab3_model_0039.pth')

# Test model
_, test_top1 = test_network(test_loader, main_model)

print("Epoch {:1.0f}, Accuracy (test set) {:2.3f}%".format(num_epoch, test_top1))

Epoch 39, Accuracy (test set) 96.310%


**5. Контрольные вопросы**

1. Что такое верификация и идентицикация диктора?

2. Что такое распознавание диктора на закрытом и открытом множестве?

3. Что такое текстозависимое и текстонезависимое распознавание диктора?

4. Описать схему обучения блока генерации дикторских моделей на основе нейронных сетей.

5. Описать основные компоненты, из которых состоит нейросетевой блок генерации дикторских моделей (фреймовый уровень, слой статистического пулинга, сегментный уровень, выходной слой).

6. Как устроены нейросетевые архитектуры на основе ResNet-блоков?

7. Что такое полносвязная нейронная сеть прямого распространения?

8. Как устроена стоимостная функция для обучения нейросетевого блока генерации дикторских моделей?

9. Что такое аугментация данных?

10. Что такое дикторский эмбеддинг и на каком уровне блока построения дикторских моделей он генерируется?

**Ответы на контрольные вопросы:**

1. **Что такое верификация и идентификация диктора?**

   **Верификация диктора** (speaker verification) — задача проверки, является ли предъявленный голос голосом конкретного заявленного диктора. Это задача бинарной классификации: "да" или "нет". Система сравнивает тестовый эмбеддинг с эталонным эмбеддингом заявленного диктора и принимает решение о совпадении или несовпадении.

   **Идентификация диктора** (speaker identification) — задача определения, какому диктору из известного множества принадлежит предъявленный голос. Это задача многоклассовой классификации, где система выбирает наиболее вероятного диктора из закрытого или открытого множества.

2. **Что такое распознавание диктора на закрытом и открытом множестве?**

   **Распознавание на закрытом множестве** (closed-set identification) — задача идентификации, при которой все возможные дикторы заранее известны и присутствуют в обучающей выборке. Количество дикторских классов строго фиксировано. В лабораторной работе используется именно этот подход: модель обучается на фиксированном множестве дикторов из VoxCeleb1 dev set, и тестирование происходит на этом же множестве.

   **Распознавание на открытом множестве** (open-set identification) — задача идентификации, при которой тестовый диктор может не принадлежать ни одному из известных классов. Система должна либо определить наиболее вероятного диктора из известных, либо отклонить запрос как неизвестного диктора.

3. **Что такое текстозависимое и текстонезависимое распознавание диктора?**

   **Текстозависимое распознавание** (text-dependent) — система требует, чтобы тестовый диктор произносил те же фразы или слова, что использовались при регистрации. Это упрощает задачу, но ограничивает практическое применение.

   **Текстонезависимое распознавание** (text-independent) — система может распознавать диктора независимо от произносимого текста. В лабораторной работе используется текстонезависимый подход: модель обучается на произвольных фрагментах речи из VoxCeleb1, которые могут содержать любые слова и фразы.

4. **Описать схему обучения блока генерации дикторских моделей на основе нейронных сетей.**

   Схема обучения включает следующие этапы:

   - **Подготовка данных**: генерация списков тренировочных, валидационных и тестовых данных на основе протокола идентификации (iden_split.txt), исключение дикторов из test set для корректного тестирования.

   - **Извлечение акустических признаков**: для каждой звукозаписи вычисляются логарифмы энергий на выходе мел-банка фильтров (40 мел-фильтров, частота дискретизации 16 кГц), выполняется нормализация признаков.

   - **Инициализация модели**: создание архитектуры ResNet с заданными параметрами (layers=[3,4,6,3], num_filters=[32,64,128,256], nOut=512), инициализация функции потерь AAMSoftmaxLoss с параметрами margin=0.35 и scale=32.0.

   - **Обучение**: процесс обучения выполняется в цикле по эпохам (max_epoch=40). На каждой итерации:
     * Формируются мини-батчи коротких фрагментов (max_frames_train=200) из различных дикторов
     * Применяется аугментация данных (опционально): реверберация, аддитивный шум (речь, музыка)
     * Вычисляется прямой проход через модель (forward pass)
     * Вычисляется функция потерь AAMSoftmaxLoss
     * Выполняется обратное распространение ошибки (backward pass)
     * Обновляются параметры модели с помощью оптимизатора SGD
     * Обновляется learning rate с помощью планировщика OneCycleLR

   - **Валидация**: каждые val_interval=5 эпох выполняется валидация на валидационном множестве для контроля переобучения.

   - **Сохранение модели**: периодическое сохранение параметров модели для возможности возобновления обучения.

5. **Описать основные компоненты, из которых состоит нейросетевой блок генерации дикторских моделей (фреймовый уровень, слой статистического пулинга, сегментный уровень, выходной слой).**

   Архитектура блока генерации дикторских моделей состоит из четырёх уровней:

   **Фреймовый уровень** (Frame-level): 
   - Предназначен для формирования локальных представлений голоса
   - Состоит из начальной свёртки (conv1, 3x3, 32 фильтра), батч-нормализации и активации ReLU
   - Содержит 4 слоя ResNet-блоков (layer1-layer4) с возрастающим количеством фильтров [32, 64, 128, 256] и страйдом [1, 2, 2, 2]
   - Каждый слой содержит соответственно [3, 4, 6, 3] ResNet-блока типа BasicBlock
   - Выходом являются карты признаков с локальными особенностями голоса

   **Слой статистического пулинга** (Statistical Pooling):
   - Удаляет временную размерность, формируя вектор фиксированной длины
   - В классическом варианте (SP): вычисляет среднее (mu) и стандартное отклонение (sg) карт признаков вдоль временной оси, затем конкатенирует их
   - В варианте с вниманием (ASP): использует механизм внимания для взвешенного усреднения, затем вычисляет взвешенное среднее и стандартное отклонение
   - Выход: вектор размерности num_filters[3]*outmap_size*2 (для SP и ASP)

   **Сегментный уровень** (Segment-level):
   - Трансформирует промежуточный вектор высокой размерности в компактный дикторский эмбеддинг
   - Состоит из полносвязного слоя MaxoutLinear (два линейных слоя с операцией максимума) и батч-нормализации без аффинного преобразования
   - Выход: дикторский эмбеддинг размерности nOut=512

   **Выходной слой** (Output layer):
   - Полносвязный слой с softmax-активацией
   - Количество выходов равно числу дикторов в тренировочной выборке (num_train_spk)
   - Используется только на этапе обучения для вычисления функции потерь
   - На этапе тестирования не используется (используются только первые три уровня)

6. **Как устроены нейросетевые архитектуры на основе ResNet-блоков?**

   ResNet-блоки построены на основе принципа остаточных связей (residual connections):

   **BasicBlock** (используется в лабораторной работе):
   - Состоит из двух свёрточных слоев 3x3 с батч-нормализацией и активацией ReLU между ними
   - Содержит остаточное соединение (skip connection): вход блока складывается с выходом второго свёрточного слоя
   - Если размерность входных и выходных данных не совпадает (stride != 1 или изменение числа каналов), используется downsample-слой (1x1 свёртка с батч-нормализацией) для приведения размерности входа к размерности выхода
   - Формула: out = ReLU(conv2(ReLU(conv1(x))) + downsample(x))

   **Архитектура ResNet в лабораторной работе**:
   - Начальный слой: conv1 (3x3, 32 фильтра) → BatchNorm → ReLU
   - 4 уровня ResNet-блоков:
     * layer1: 3 блока, 32 фильтра, stride=1
     * layer2: 4 блока, 64 фильтра, stride=2
     * layer3: 6 блоков, 128 фильтров, stride=2
     * layer4: 3 блока, 256 фильтров, stride=2
   - Каждый уровень уменьшает пространственную размерность и увеличивает количество каналов
   - Остаточные связи позволяют эффективно обучать глубокие сети, решая проблему затухающих градиентов

7. **Что такое полносвязная нейронная сеть прямого распространения?**

   Полносвязная нейронная сеть прямого распространения (fully connected feedforward neural network) — это тип нейронной сети, в которой каждый нейрон одного слоя соединён со всеми нейронами следующего слоя.

   **Характеристики**:
   - Прямое распространение: информация движется только вперёд от входного слоя через скрытые слои к выходному слою (без обратных связей)
   - Полная связность: каждый нейрон получает вход от всех нейронов предыдущего слоя
   - Математически: выход слоя вычисляется как y = f(Wx + b), где W — матрица весов, x — входной вектор, b — вектор смещений, f — функция активации

   **В лабораторной работе**:
   - На сегментном уровне используется MaxoutLinear: два параллельных полносвязных слоя с операцией максимума
   - Выходной слой (в функции потерь AAMSoftmaxLoss) содержит полносвязный слой, который преобразует эмбеддинг размерности 512 в вектор размерности num_train_spk (количество дикторов)

8. **Как устроена стоимостная функция для обучения нейросетевого блока генерации дикторских моделей?**

   В лабораторной работе используется функция потерь **AAMSoftmaxLoss** (Additive Angular Margin Softmax Loss):

   **Параметры**:
   - margin (m) = 0.35 — угловой зазор, добавляемый для увеличения разделимости классов
   - scale (s) = 32.0 — масштабирующий параметр

   **Алгоритм работы**:
   1. Нормализация входного эмбеддинга x и весов W (L2-нормализация)
   2. Вычисление косинуса угла между эмбеддингом и весами: cosine = F.linear(F.normalize(x), F.normalize(self.weight))
   3. Вычисление cos(θ + m) с использованием тригонометрических тождеств:
      - sine = sqrt(1 - cosine²)
      - phi = cosine*cos(m) - sine*sin(m)
   4. Применение условия для обеспечения монотонности функции
   5. Создание one-hot вектора для правильного класса
   6. Формирование выходного вектора: output = (one_hot*phi) + ((1.0 - one_hot)*cosine)
   7. Масштабирование: output = output * scale
   8. Вычисление категориальной кросс-энтропии: loss = CrossEntropyLoss(output, label)

   **Преимущества**: AAMSoftmaxLoss увеличивает угловое расстояние между классами, что улучшает разделимость дикторских эмбеддингов в пространстве признаков.

9. **Что такое аугментация данных?**

   **Аугментация данных** (data augmentation) — методика создания дополнительных обучающих примеров из имеющихся данных путём внесения в них искажений, которые могут потенциально возникнуть на этапе итогового тестирования системы.

   **В лабораторной работе** используются следующие типы аугментации:

   - **Реверберация** (reverberation): моделирование импульсного отклика помещения путём свёртки исходного сигнала с импульсным откликом комнаты (RIR) из базы SLR28. Реализовано в методе `reverberate()` класса `AugmentWAV`.

   - **Аддитивный шум речи** (additive speech noise): добавление фонового шума голосов нескольких дикторов из базы MUSAN (SLR17) с SNR в диапазоне [13, 20] дБ. Количество источников шума: от 3 до 7.

   - **Аддитивный музыкальный шум** (additive music noise): добавление музыкального шума из базы MUSAN с SNR в диапазоне [5, 15] дБ.

   - **Аддитивный неструктурированный шум** (additive noise): добавление неструктурированного шума из базы MUSAN с SNR в диапазоне [0, 15] дБ.

   **Реализация**: Аугментация выполняется онлайн (во время обучения) с вероятностью, определяемой случайным выбором типа искажения. Это увеличивает разнообразие тренировочных данных и улучшает устойчивость модели к искажениям.

10. **Что такое дикторский эмбеддинг и на каком уровне блока построения дикторских моделей он генерируется?**

    **Дикторский эмбеддинг** (speaker embedding) — это высокоуровневый вектор-признаков фиксированной размерности (например, 128, 256, 512 значений), содержащий компактное представление особенностей голоса конкретного человека. Эмбеддинг извлекает инвариантные к тексту характеристики голоса, которые позволяют различать разных дикторов.

    **Генерация эмбеддинга**:
    - Дикторский эмбеддинг генерируется на **сегментном уровне** блока построения дикторских моделей
    - После прохождения через фреймовый уровень (ResNet-блоки) и слой статистического пулинга, промежуточный вектор высокой размерности поступает на сегментный уровень
    - Сегментный уровень состоит из полносвязного слоя MaxoutLinear, который трансформирует промежуточный вектор в компактный эмбеддинг размерности nOut=512
    - Выход сегментного уровня и является дикторским эмбеддингом

    **Использование**:
    - **Эталонные эмбеддинги**: формируются на этапе регистрации дикторской модели и хранятся в базе данных
    - **Тестовые эмбеддинги**: формируются на этапе использования системы при предъявлении голоса пользователя
    - Сравнение эмбеддингов выполняется с помощью метрики косинусного расстояния или евклидова расстояния для принятия решения о верификации или идентификации диктора


**6. Список литературы**

1. Bai Z., Zhang X.-L., Chen J. Speaker recognition based on deep learning: an overview // 	arXiv:2012.00931 [eess.AS] ([ссылка](https://arxiv.org/pdf/2012.00931.pdf)).

2. Hansen J.H.L., Hasan T. Speaker recognition by machines and humans: a tutorial review // IEEE Signal Processing Magazine, 2015. V. 32. № 6. P. 74–99 ([ссылка](https://www.researchgate.net/publication/282940395_Speaker_Recognition_by_Machines_and_Humans_A_tutorial_review)).